In [1]:
# Imports
import polars as pl
from pathlib import Path
import os
os.chdir('/Users/michaelmayo/Downloads/data')

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "dat_train1.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "dat_train1_clean.csv"

df = pl.scan_csv(DATA_PATH)

In [2]:
## task 1

#1.1
num_rows = df.select(pl.len()).collect().item()
#1.2
num_unique_ids = df.select(pl.col("id").n_unique()).collect().item()

#1.3
earliest, latest = df.select([
    pl.col("event_timestamp").min().alias("earliest"),
    pl.col("event_timestamp").max().alias("latest")
]).collect().row(0)



print("Task 1.1 - Number of rows:", num_rows)

print("Task 1.2 - Number of unique IDs:", num_unique_ids)

print("Task 1.3 - Earliest timestamp:", earliest)
print("Task 1.3 - Latest timestamp:", latest)

Task 1.1 - Number of rows: 54960961
Task 1.2 - Number of unique IDs: 1430445
Task 1.3 - Earliest timestamp: 2020-11-03T03:31:30Z
Task 1.3 - Latest timestamp: 2023-01-23T12:29:56Z


In [3]:
import polars as pl

def create_journey_tables(input_csv_path):
    q = pl.scan_csv(input_csv_path)

    # keep your duplicate removal idea
    q = q.unique(subset=["id", "event_name", "event_timestamp"])

    # parse timestamps
    q = q.with_columns(
        pl.col("event_timestamp").str.to_datetime(time_zone="UTC")
    )

    # sort within journey
    q = q.sort(["id", "event_timestamp", "ed_id"])

    # get latest timestamp in whole dataset
    max_time = q.select(pl.col("event_timestamp").max()).collect().item()

    # build one-row-per-journey table
    journeys = (
        q.group_by("id")
        .agg([
            pl.struct(["event_timestamp", "ed_id"]).alias("journey"),

            pl.len().alias("num_actions"),
            pl.col("ed_id").n_unique().alias("num_unique_actions"),

            pl.col("event_timestamp").min().alias("start_time"),
            pl.col("event_timestamp").max().alias("end_time"),

            (pl.col("event_timestamp").max() - pl.col("event_timestamp").min())
            .dt.total_seconds()
            .alias("duration_seconds"),

            pl.col("ed_id").first().alias("first_action"),
            pl.col("ed_id").last().alias("last_action"),

            pl.col("event_name").eq("order_shipped").any().alias("has_order_shipped"),
        ])
        .with_columns([
            ((pl.lit(max_time) - pl.col("end_time")) > pl.duration(days=60))
            .alias("older_than_60_days")
        ])
    ).collect()

    # split into three tables
    complete = journeys.filter(
        pl.col("has_order_shipped")
    )

    incomplete = journeys.filter(
        (~pl.col("has_order_shipped")) & (pl.col("older_than_60_days"))
    )

    ongoing = journeys.filter(
        (~pl.col("has_order_shipped")) & (~pl.col("older_than_60_days"))
    )

    return ongoing, incomplete, complete

In [4]:
training_csv_path = "~/Downloads/data/dat_train1.csv"

ongoing, incomplete, complete = create_journey_tables(training_csv_path)

print("ONGOING")
print(ongoing.head())

print("INCOMPLETE")
print(incomplete.head())

print("COMPLETE")
print(complete.head())

print("Counts:")
print("ongoing:", ongoing.height)
print("incomplete:", incomplete.height)
print("complete:", complete.height)

ONGOING
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ journey   ┆ num_actio ┆ num_uniqu ┆ … ┆ first_act ┆ last_acti ┆ has_order ┆ older_th │
│ ---       ┆ ---       ┆ ns        ┆ e_actions ┆   ┆ ion       ┆ on        ┆ _shipped  ┆ an_60_da │
│ str       ┆ list[stru ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ys       │
│           ┆ ct[2]]    ┆ u32       ┆ u32       ┆   ┆ i64       ┆ i64       ┆ bool      ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ bool     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ -62139864 ┆ [{2022-01 ┆ 23        ┆ 6         ┆ … ┆ 12        ┆ 24        ┆ false     ┆ false    │
│ 7 -840300 ┆ -06       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 309       ┆ 19:04:53  ┆           ┆           ┆   ┆           ┆   

In [5]:
import polars as pl
import random
import math

# combine incomplete + complete, truncate once, then split back out
def truncate_combined_table(combined_table, seed=42, max_samples_per_journey=60):
    random.seed(seed)
    truncated_rows = []

    for row in combined_table.iter_rows(named=True):
        journey_id = row["id"]
        journey = row["journey"]
        start_time = row["start_time"]
        end_time = row["end_time"]
        source_table = row["source_table"]   # "incomplete" or "complete"

        if journey is None or len(journey) == 0 or start_time is None or end_time is None:
            continue

        start_ts = start_time.timestamp()
        end_ts = end_time.timestamp()

        duration_days = (end_ts - start_ts) / 86400
        n_samples = max(1, math.ceil(duration_days))
        n_samples = min(n_samples, max_samples_per_journey)

        # pull these out once so we do not keep re-reading structs
        event_times = [e["event_timestamp"].timestamp() for e in journey]
        event_ids = [e["ed_id"] for e in journey]

        for sample_num in range(1, n_samples + 1):
            cutoff_ts = random.uniform(start_ts, end_ts)

            keep_idx = 0
            for t in event_times:
                if t <= cutoff_ts:
                    keep_idx += 1
                else:
                    break

            if keep_idx == 0:
                continue

            first_time = journey[0]["event_timestamp"]
            last_time = journey[keep_idx - 1]["event_timestamp"]

            truncated_rows.append({
                "id": journey_id,
                "sample_num": sample_num,
                "source_table": source_table,
                "num_actions": keep_idx,
                "num_unique_actions": len(set(event_ids[:keep_idx])),
                "start_time": first_time,
                "end_time": last_time,
                "duration_seconds": int((last_time - first_time).total_seconds()) if keep_idx > 1 else 0,
                "first_action": event_ids[0],
                "last_action": event_ids[keep_idx - 1],
            })

    return pl.DataFrame(truncated_rows)

In [6]:
# add a column telling us where each row came from
incomplete_labeled = incomplete.with_columns(
    pl.lit("incomplete").alias("source_table")
)

complete_labeled = complete.with_columns(
    pl.lit("complete").alias("source_table")
)

# combine
combined = pl.concat([incomplete_labeled, complete_labeled], how="vertical")

# practical move: sample first so it runs faster
combined_small = combined.sample(n=5000, shuffle=True, seed=42)

# truncate once
combined_truncate = truncate_combined_table(
    combined_small,
    seed=42,
    max_samples_per_journey=60
)

# split back apart
incomplete_truncate = combined_truncate.filter(
    pl.col("source_table") == "incomplete"
)

complete_truncate = combined_truncate.filter(
    pl.col("source_table") == "complete"
)

print("combined_small:", combined_small.height)
print("combined_truncate:", combined_truncate.height)
print("incomplete_truncate:", incomplete_truncate.height)
print("complete_truncate:", complete_truncate.height)

print(incomplete_truncate.head())
print(complete_truncate.head())

combined_small: 5000
combined_truncate: 214345
incomplete_truncate: 183727
complete_truncate: 30618
shape: (5, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ sample_nu ┆ source_ta ┆ num_actio ┆ … ┆ end_time  ┆ duration_ ┆ first_act ┆ last_act │
│ ---       ┆ m         ┆ ble       ┆ ns        ┆   ┆ ---       ┆ seconds   ┆ ion       ┆ ion      │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ datetime[ ┆ ---       ┆ ---       ┆ ---      │
│           ┆ i64       ┆ str       ┆ i64       ┆   ┆ μs, UTC]  ┆ i64       ┆ i64       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 974955308 ┆ 1         ┆ incomplet ┆ 50        ┆ … ┆ 2021-09-1 ┆ 14413382  ┆ 2         ┆ 1        │
│ -48500392 ┆           ┆ e         ┆           ┆   ┆ 6         ┆           ┆           ┆          │
│ 1         ┆           ┆           ┆           ┆   ┆ 01:43:02  ┆           ┆